# LightGBM and XGBoost: Titanic Dataset


## Assignment Explanation

### Objective
The objective of this assignment is to compare XGBoost and LightGBM models using the Titanic dataset.

### Methodology
The Titanic data is preprocessed by removing identity columns, imputing missing values, and encoding categorical variables. The data is split into training and testing sets. XGBoost and LightGBM models are trained and evaluated using classification metrics and ROC-AUC.

### Interpretation
Both XGBoost and LightGBM are gradient boosting algorithms. XGBoost is known for strong regularization and accuracy, while LightGBM is designed for faster training. Comparing their metrics helps decide which model performs better on the Titanic survival prediction task.


In [3]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style='whitegrid')
pd.set_option('display.max_columns', None)


In [ ]:
%pip install xgboost
%pip install lightgbm

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

train = pd.read_csv(r'C:\Users\lenovo\Downloads\Assignments\XGBM & LGBM\XGBM & LGBM\Titanic_train.csv')
train.head()


ModuleNotFoundError: No module named 'xgboost'

In [ ]:
target = 'Survived'
drop_cols = [c for c in ['PassengerId', 'Name', 'Ticket', 'Cabin'] if c in train.columns]
X = train.drop(columns=[target] + drop_cols)
y = train[target]
num_cols = X.select_dtypes(include=np.number).columns
cat_cols = X.columns.difference(num_cols)
preprocess = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:
models = {
    'XGBoost': XGBClassifier(eval_metric='logloss', random_state=42),
    'LightGBM': LGBMClassifier(random_state=42)
}
for name, clf in models.items():
    pipe = Pipeline([('prep', preprocess), ('model', clf)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]
    print('\\n', name)
    print(classification_report(y_test, pred))
    print('ROC-AUC:', roc_auc_score(y_test, proba))
